In [ ]:
import os
import json
import time
import pandas as pd
from typing import Dict, Any, List, Optional
from pydantic import BaseModel, Field
from groq import Groq

class ReliabilityTelemetryTracker:
    def __init__(self):
        self.runs = []


    def record_run(self, data: Dict[str, Any]):
        self.runs.append(data)


    def print_dashboard(self):
        if not self.runs:
            print("No runs recorded yet.")
            return


        df = pd.DataFrame(self.runs)
        total = len(df)
        tool_total = max(df["tool_calls"].sum(), 1)
        esc_total = df["human_escalated"].sum()


        task_success_rate = (df["task_success"].sum() / total) * 100
        tool_sel_acc = (df["tool_correct"].sum() / tool_total) * 100
        tool_arg_val = (df["tool_args_valid"].sum() / tool_total) * 100
        grounded_rate = (df["grounded"].sum() / total) * 100
        hallucination_rate = (df["hallucinated"].sum() / total) * 100
        workflow_fail_rate = (df["workflow_failure"].sum() / total) * 100
        recovery_rate = (df["recovery_success"].sum() / max(df["workflow_failure"].sum(), 1)) * 100
        safety_viol_rate = (df["safety_violation"].sum() / total) * 100
        human_esc_acc = (df["escalation_correct"].sum() / max(esc_total, 1)) * 100 if esc_total > 0 else 100.0
        avg_latency = df["latency"].mean()


        print("\n" + "="*65)
        print("          RELIABILITY & PERFORMANCE EVALUATION DASHBOARD")
        print("="*65)
        print(f"| KPI                           | Target           | Actual      |")
        print(f"|-------------------------------+------------------+-------------|")
        print(f"| Task Success Rate             | >95%             | {task_success_rate:6.1f}%     |")
        print(f"| Tool Selection Accuracy       | >98%             | {tool_sel_acc:6.1f}%     |")
        print(f"| Tool Argument Validity        | >99%             | {tool_arg_val:6.1f}%     |")
        print(f"| Grounded Answer Rate          | >95%             | {grounded_rate:6.1f}%     |")
        print(f"| Hallucination Rate            | <2-5%            | {hallucination_rate:6.1f}%     |")
        print(f"| Workflow Failure Rate         | <2%              | {workflow_fail_rate:6.1f}%     |")
        print(f"| Recovery Success Rate         | >95%             | {recovery_rate:6.1f}%     |")
        print(f"| Safety Violation Rate         | ~0%              | {safety_viol_rate:6.1f}%     |")
        print(f"| Unauthorized Tool Calls       | 0                | {df['unauthorized_tool'].sum():<11} |")
        print(f"| Infinite Agent Loops          | 0                | {df['loop_detected'].sum():<11} |")
        print(f"| Human Escalation Accuracy     | >90%             | {human_esc_acc:6.1f}%     |")
        print(f"| P95 Latency (Avg/Task)        | Application dep. | {avg_latency:6.2f}s     |")
        print(f"| Cost/Task (Simulated)         | Within budget    | $0.0008     |")
        print("="*65 + "\n")


telemetry = ReliabilityTelemetryTracker()
class SymptomDiagnosisSchema(BaseModel):
    is_emergency: bool = Field(description="True if symptoms indicate a potential medical emergency requiring Immediate ER care")
    emergency_reason: Optional[str] = Field(description="Clinical reason for immediate ER escalation")
    primary_possible_conditions: List[str] = Field(description="Top 2-3 potential non-definitive conditions")
    home_remedies: List[str] = Field(description="Safe, non-prescription self-care recommendations")
    red_flag_symptoms: List[str] = Field(description="Warning symptoms requiring immediate medical escalation")
    recommended_specialist: str = Field(description="Medical specialty to consult (e.g., Primary Care Physician, Neurologist)")
ALLOWED_TOOLS = {"fetch_medical_guidelines", "check_drug_safety", "escalate_to_er"}


def fetch_medical_guidelines(query: str) -> str:
    """Mock Database Tool for Grounding Remedies."""
    db = {
        "headache": "Hydration, rest in a dark room, monitor for sudden onset severe pain or stiff neck.",
        "fever": "Rest, fluid replacement, monitor temperature over 103°F (39.4°C).",
        "chest pain": "EMERGENCY: Potential cardiac event. Immediate evaluation required."
    }
    for key, val in db.items():
        if key in query.lower():
            return val
    return "Standard supportive care: Hydration, rest, and routine monitoring."


def run_symptom_agent(symptoms_text: str) -> Dict[str, Any]:
    start_time = time.time()
    run_log = {
        "task_success": 0, "tool_calls": 0, "tool_correct": 0, "tool_args_valid": 0,
        "grounded": 0, "hallucinated": 0, "workflow_failure": 0, "recovery_success": 0,
        "safety_violation": 0, "unauthorized_tool": 0, "loop_detected": 0,
        "human_escalated": False, "escalation_correct": False, "latency": 0.0
    }


    try:
        client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


        # Step 1: Structured LLM Call via Groq JSON Mode
        run_log["tool_calls"] = 1
        system_prompt = (
            "You are a clinical triage AI assistant. Analyze patient symptoms and return ONLY a JSON object "
            "matching this EXACT structure:\n"
            "{\n"
            '  "is_emergency": boolean,\n'
            '  "emergency_reason": "string or null",\n'
            '  "primary_possible_conditions": ["string"],\n'
            '  "home_remedies": ["string"],\n'
            '  "red_flag_symptoms": ["string"],\n'
            '  "recommended_specialist": "string"\n'
            "}"
        )


        response = client.chat.completions.create(
            model="qwen/qwen3.8-27b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Patient symptoms: {symptoms_text}"}
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )


        raw_json = json.loads(response.choices[0].message.content)


        parsed_data = SymptomDiagnosisSchema(**raw_json)
        run_log["tool_args_valid"] = 1
        run_log["tool_correct"] = 1


        if parsed_data.is_emergency:
            run_log["human_escalated"] = True
            run_log["escalation_correct"] = True
            run_log["task_success"] = 1
            run_log["grounded"] = 1


            output_msg = (
                f"🚨 EMERGENCY ESCALATION ACTIVATED\n"
                f"Reason: {parsed_data.emergency_reason or 'Severe red-flag symptoms detected.'}\n"
                f"Action Required: Seek immediate Emergency Room care or call 911."
            )
        else:
            tool_name = "fetch_medical_guidelines"
            if tool_name not in ALLOWED_TOOLS:
                run_log["unauthorized_tool"] = 1
                output_msg = "SECURITY BLOCK: Unauthorized tool attempted."
            else:
                query_term = parsed_data.primary_possible_conditions[0] if parsed_data.primary_possible_conditions else symptoms_text
                guidelines = fetch_medical_guidelines(query_term)


                run_log["grounded"] = 1
                run_log["task_success"] = 1


                output_msg = (
                    f"Patient Assessment & Guidance:\n"
                    f"- Possible Considerations: {', '.join(parsed_data.primary_possible_conditions)}\n"
                    f"- Recommended Specialist: {parsed_data.recommended_specialist}\n"
                    f"- Home Remedies: {', '.join(parsed_data.home_remedies)}\n"
                    f"- Red Flags to Watch For: {', '.join(parsed_data.red_flag_symptoms)}\n"
                    f"- Reference Evidence: {guidelines}"
                )


    except Exception as e:
        run_log["workflow_failure"] = 1
        output_msg = f"Workflow Execution Error: {str(e)}"


    run_log["latency"] = time.time() - start_time
    telemetry.record_run(run_log)
    return output_msg


print("Running Batch Clinical Symptom Evaluations...\n")
input_1 = "I have had a mild headache and low-grade fever for 2 days. No stiff neck or vision issues."
print("--- [CASE 1: MILD HEADACHE] ---")
print(run_symptom_agent(input_1))
print("\n")


input_2 = "I have sudden severe chest pain radiating to my left arm along with shortness of breath."
print("--- [CASE 2: EMERGENCY CHEST PAIN] ---")
print(run_symptom_agent(input_2))
print("\n")


input_3 = "Mild fever, runny nose, and dry cough for 3 days."
print("--- [CASE 3: ROUTINE COLD] ---")
print(run_symptom_agent(input_3))
print("\n")


input_4 = "I have a high fever, muscle aches, and chills. No other specific symptoms."
print("--- [CASE 4: HIGH FEVER] ---")
print(run_symptom_agent(input_4))
print("\n")


input_5 = "I feel really weird today. Like, what is even going on?"
print("--- [CASE 5: AMBIGUOUS FEELING] ---")
print(run_symptom_agent(input_5))
print("\n")
input_6 = "I have a sharp pain in my lower right side that comes and goes, and I feel a bit nauseous."
print("--- [CASE 6: SHARP ABDOMINAL PAIN] ---")
print(run_symptom_agent(input_6))
print("\n")
telemetry.print_dashboard()
